# 00 — Train trên Google Colab

**Dùng notebook này để train M3** (3 backbone pretrained ở 224×224).
T4 của Colab nhanh hơn MPS của MacBook khoảng **3–4 lần** ở độ phân giải này.

M1 và M2 chạy ở 48×48 nên **không cần Colab** — train ngay trên máy cá nhân nhanh hơn
(khỏi mất công upload).

---

## Chạy theo đúng thứ tự ô bên dưới

| Ô | Việc | Thời gian |
|---|---|---|
| 1 | Kiểm tra có GPU không | 5 giây |
| 2 | Kết nối Google Drive (để checkpoint sống sót khi mất session) | 30 giây |
| 3 | Đưa code lên Colab | 1 phút |
| 4 | Cài gói còn thiếu | 30 giây |
| 5 | Tải + tiền xử lý GTSRB | ~4 phút |
| 6 | **Train M3** | ~15 phút/backbone |
| 7 | Nếu mất session → chạy tiếp | — |
| 8 | Tải kết quả về máy | 1 phút |

> ⚠️ **Bật GPU trước khi chạy:** menu `Runtime` → `Change runtime type` → `T4 GPU` → `Save`.

## Ô 1 — Kiểm tra GPU

Nếu ô này in `CUDA: False` thì bạn **chưa bật GPU**. Quay lại
`Runtime → Change runtime type → T4 GPU` rồi chạy lại.

In [ ]:
import torch, subprocess

print("torch     :", torch.__version__)
print("CUDA      :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU       :", torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM      : {total_gb:.1f} GB")
else:
    raise SystemExit(
        "CHƯA BẬT GPU.\n"
        "Runtime -> Change runtime type -> T4 GPU -> Save, rồi chạy lại ô này.\n"
        "Train M3 trên CPU mất khoảng 8 tiếng thay vì 15 phút."
    )

print()
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## Ô 2 — Kết nối Google Drive

★ **Đây là ô quan trọng nhất.** Colab miễn phí hay ngắt session giữa chừng. Nếu checkpoint
nằm trong `/content` thì mất hết. Ta cho `artifacts/` trỏ thẳng sang Drive, nên dù mất
session vẫn `--resume` được (xem Ô 7).

Chạy ô này sẽ hiện cửa sổ xin quyền — bấm đồng ý.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Thư mục lưu trữ lâu dài trên Drive
DRIVE_DIR = Path("/content/drive/MyDrive/gtsrb")
(DRIVE_DIR / "artifacts" / "runs").mkdir(parents=True, exist_ok=True)

print("Thư mục trên Drive:", DRIVE_DIR)
print("Các run đã có:")
runs = sorted((DRIVE_DIR / "artifacts" / "runs").glob("*"))
for r in runs:
    print("  ", r.name)
if not runs:
    print("   (chưa có run nào — bình thường nếu đây là lần đầu)")

## Ô 3 — Đưa code lên Colab

**Hai cách, chọn một.**

### Cách A — Upload file zip *(nhanh nhất, không cần GitHub)*

Trên máy bạn chạy:
```bash
cd ~/Downloads/DLprj
make zip            # tạo gtsrb_code.zip, chỉ ~400 KB
```
Rồi chạy ô bên dưới và chọn file `gtsrb_code.zip` khi nó hiện nút upload.

### Cách B — Clone từ GitHub *(tiện hơn nếu nhóm dùng chung repo)*

Thay ô dưới bằng:
```python
!git clone https://github.com/<tên-nhóm>/<tên-repo>.git /content/DLprj
%cd /content/DLprj
```

In [ ]:
# ---- Cách A: upload zip ----
import os, zipfile, shutil
from pathlib import Path

PROJECT = Path("/content/DLprj")

if PROJECT.exists():
    print("Đã có code ở", PROJECT, "- bỏ qua bước upload.")
    print("(Muốn upload lại thì chạy:  !rm -rf /content/DLprj  rồi chạy lại ô này)")
else:
    from google.colab import files
    print("Chọn file gtsrb_code.zip (tạo bằng `make zip` trên máy bạn):")
    uploaded = files.upload()

    zip_name = next(iter(uploaded))
    PROJECT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_name) as archive:
        archive.extractall(PROJECT)
    os.remove(zip_name)
    print("Đã giải nén vào", PROJECT)

os.chdir(PROJECT)
print("Thư mục làm việc:", os.getcwd())
print("Nội dung:", sorted(p.name for p in PROJECT.iterdir())[:12])

## Ô 4 — Cài gói còn thiếu + trỏ `artifacts/` sang Drive

Chỉ cài `thop`. Mọi gói khác Colab đã có sẵn.

★ **Không bao giờ `pip install torch` trên Colab** — sẽ kéo về bản không khớp CUDA và
bạn train trên CPU mà không biết.

In [ ]:
!pip install -q -r requirements-colab.txt

# ---- Trỏ artifacts/ sang Drive để checkpoint sống sót khi mất session ----
import os, shutil
from pathlib import Path

local_artifacts = Path("/content/DLprj/artifacts")
drive_artifacts = Path("/content/drive/MyDrive/gtsrb/artifacts")

if local_artifacts.is_symlink():
    print("artifacts/ đã trỏ sang Drive rồi.")
else:
    if local_artifacts.exists():
        shutil.rmtree(local_artifacts)
    local_artifacts.symlink_to(drive_artifacts)
    print("artifacts/ ->", drive_artifacts)

# Kiểm tra: ghi thử một file rồi xoá
probe = local_artifacts / ".probe"
probe.write_text("ok")
assert (drive_artifacts / ".probe").read_text() == "ok", "symlink không hoạt động"
probe.unlink()
print("Đã kiểm tra: ghi vào artifacts/ thật sự lưu lên Drive.")

import thop, torch
print("thop", thop.__version__, "| torch", torch.__version__)

## Ô 5 — Tải và tiền xử lý GTSRB

Tải thẳng từ archive gốc (nhanh hơn upload từ máy bạn rất nhiều — mạng Colab khoảng
100 MB/s).

**Lưu ý:** dữ liệu để ở `/content` chứ **không** để trên Drive. Nó 1,1 GB và tải lại chỉ
mất 4 phút, trong khi đọc từ Drive trong lúc train thì **chậm hơn nhiều** (Drive gắn qua
mạng). Chỉ checkpoint mới đáng để trên Drive.

In [ ]:
!python scripts/prepare_data.py --download --preprocess clahe --img-size 48

Ô dưới xác nhận hai con số **phải kiểm bằng mắt** trước khi train bất cứ thứ gì.

In [ ]:
import sys; sys.path.insert(0, "src")
import pandas as pd
from gtsrb.data.split import check_leakage

frame = pd.read_csv("data/processed/index.csv")
report = check_leakage(frame)
counts = frame["split"].value_counts()

print(f"Tổng ảnh          : {len(frame)}          (phải là 51839)")
print(f"train/val/test    : {counts['train']} / {counts['val']} / {counts['test']}")
print(f"Track dùng chung  : {report['n_shared_tracks']}              (PHẢI là 0)")
print(f"Lệch phân phối lớp: {report['max_class_ratio_gap_pct']:.3f}%         (phải < 1%)")

assert len(frame) == 51839, "Thiếu ảnh — xem docs/SU_CO.md mục 1"
assert report["n_shared_tracks"] == 0, "CÓ RÒ RỈ DỮ LIỆU — dừng lại, đừng train"
print("\nOK, dữ liệu sạch. Train được.")

## Ô 6 — Train M3

Hai tham số chỉ bật trên Colab, **không** bật trên macOS:

| Tham số | Colab | macOS | Vì sao |
|---|---|---|---|
| `train.num_workers` | **2** | 0 | macOS dùng `fork` + OpenCV hay treo DataLoader |
| `train.batch_size` | **96** | 64 | T4 có 16 GB VRAM riêng; MPS chia sẻ 16 GB RAM với hệ thống |
| `train.amp` | tự **bật** | tự **tắt** | `fit()` tự quyết theo `device.type` — không cần làm gì |

Mỗi backbone khoảng **15 phút**. Chạy từng ô một để nếu mất session thì chỉ mất một backbone.

In [ ]:
!python scripts/train.py --config configs/m3_resnet18.yaml \
    --set train.num_workers=2 train.batch_size=96

In [ ]:
!python scripts/train.py --config configs/m3_mobilenetv2.yaml \
    --set train.num_workers=2 train.batch_size=96

In [ ]:
!python scripts/train.py --config configs/m3_effnetb0.yaml \
    --set train.num_workers=2 train.batch_size=96

## Ô 7 — Nếu mất session giữa chừng

Colab miễn phí hay ngắt. Vì checkpoint đã nằm trên Drive (Ô 2 và Ô 4), bạn **không phải
train lại từ đầu**.

Làm lại Ô 1 → 5, rồi chạy ô dưới. Nó tìm run dở dang gần nhất và chạy tiếp từ đúng
epoch đã dừng — khôi phục cả **moment của Adam, vị trí scheduler và trạng thái early
stopping**, không chỉ trọng số.

In [ ]:
from pathlib import Path
import json

runs = sorted(Path("artifacts/runs").glob("m3_*"))
unfinished = []
for run in runs:
    if not (run / "last.pt").exists():
        continue
    # Chưa có result.json nghĩa là fit() chưa chạy xong
    if not (run / "result.json").exists():
        unfinished.append(run)

if not unfinished:
    print("Không có run M3 nào dở dang. Nếu muốn train mới, quay lại Ô 6.")
    for run in runs:
        log = run / "train_log.csv"
        n = sum(1 for _ in open(log)) - 1 if log.exists() else 0
        print(f"  {run.name}: {n} epoch, đã xong")
else:
    target = unfinished[-1]
    log = target / "train_log.csv"
    done = sum(1 for _ in open(log)) - 1 if log.exists() else 0
    print(f"Run dở dang: {target.name} (đã train {done} epoch)")
    print()
    print("Chạy lệnh này để tiếp tục:")
    # Suy ra config từ tên run: m3_resnet18_s42_2026... -> configs/m3_resnet18.yaml
    backbone = "_".join(target.name.split("_")[:2])
    print(f"  !python scripts/train.py --config configs/{backbone}.yaml \\")
    print(f"      --set train.num_workers=2 train.batch_size=96 \\")
    print(f"      --resume {target}")

## Ô 8 — Tải kết quả về máy

Bạn **không** cần tải toàn bộ `artifacts/` (nặng). Chỉ cần 3 file mỗi run để
`make eval` trên máy chạy được: `best.pt`, `config.yaml`, `result.json`.

Vì chúng đã nằm trên Drive rồi, cách gọn nhất là **đồng bộ Drive về máy** (ứng dụng
Google Drive cho máy tính). Ô dưới đóng gói lại cho ai muốn tải trực tiếp.

In [ ]:
import shutil, zipfile
from pathlib import Path

# Ghi vào thư mục dự án, KHÔNG ghi cứng /content — để ô này chạy được cả
# khi test dưới máy (nơi /content không tồn tại).
out = Path("m3_runs.zip").resolve()
runs = [r for r in sorted(Path("artifacts/runs").glob("m3_*"))
        if (r / "best.pt").exists()]

if not runs:
    print("Chưa có run M3 nào hoàn tất.")
else:
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as archive:
        for run in runs:
            for name in ("best.pt", "config.yaml", "result.json", "train_log.csv"):
                path = run / name
                if path.exists():
                    archive.write(path, f"artifacts/runs/{run.name}/{name}")
    size_mb = out.stat().st_size / 1e6
    print(f"Đã đóng gói {len(runs)} run -> {out.name} ({size_mb:.0f} MB)")
    for run in runs:
        print("  ", run.name)

    from google.colab import files
    files.download(str(out))

---

## Sau khi tải về máy

Giải nén vào thư mục dự án (ghi đè `artifacts/runs/`), rồi chạy trên máy:

```bash
make eval        # bảng so sánh cả 5 model + McNemar cho mọi cặp
make speed       # latency + Pareto  — MÁY PHẢI RẢNH, đừng chạy gì khác
make gradcam     # Grad-CAM cho cả 5 model
make report      # sinh lại docs/KET_QUA.md
```

> `make speed` phải chạy trên **máy đích**, không chạy trên Colab — số latency của T4
> không nói gì về thiết bị biên. Và lúc đo phải tắt mọi thứ khác đang chạy.

## Chú ý khi báo cáo

Ghi rõ trong báo cáo rằng **M3 train trên Colab T4** còn **M1/M2 train trên M1 Pro (MPS)**.
Điều này **không** ảnh hưởng tới so sánh accuracy (cùng dữ liệu, cùng seed, cùng `fit()`),
nhưng **có** ảnh hưởng tới cột thời gian train — không được so `train_seconds` giữa hai
thiết bị khác nhau. Cột latency thì luôn đo lại trên cùng một máy nên vẫn so sánh được.